In [1]:
import os
from dotenv import load_dotenv
import pickle

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [2]:
import paddle
gpu_available  = paddle.device.is_compiled_with_cuda()
print("GPU available:", gpu_available)

GPU available: True


In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
existing_indexes

['tugas-akhir', 'compliance-gpt', 'langchain-test']

In [5]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False, use_gpu=True)
BATCH_SIZE = 250
SLEEP_TIME = 5

In [1]:
from utils.extract import process_documents
from utils.utils import load_metadata


metadata_dir = 'metadata'
metadata = load_metadata(metadata_dir)
data_all_dir = "data_pool"

all_documents = process_documents(ocr, data_all_dir, metadata)

In [22]:
with open(f'pickles/all_documents.pkl', 'wb') as f:
    pickle.dump(all_documents, f)

In [6]:
all_documents_file = open('pickles/all_documents.pkl','rb')
all_documents = pickle.load(all_documents_file)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter
from langchain_pinecone import PineconeVectorStore
from utils.llm import get_embedding_model
import time

text_splitter = MarkdownTextSplitter(chunk_size=250, chunk_overlap=50)
all_splits = text_splitter.split_documents(all_documents)

# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# all_splits = text_splitter.split_documents(all_documents)

embed_model = get_embedding_model()

def process_in_batches(documents, batch_size, index_name, embedding_model):
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        vectorstore = PineconeVectorStore.from_documents(
            documents=batch, embedding=embedding_model, index_name=index_name
        )
        time.sleep(SLEEP_TIME)  # Pause between batches

In [12]:
process_in_batches(all_splits, BATCH_SIZE, PINECONE_INDEX, embed_model)

vectorstore = PineconeVectorStore.from_existing_index(PINECONE_INDEX, embed_model)
retriever = vectorstore.as_retriever()